# 04 · Avaliação da rodada

**O que esta etapa faz:** para cada carta da rodada, mede
- **acurácia** com as amostras de validação (20% que não entram no treino), por ano: acerto global, precisão, recall e F1 da classe urbana;
- **área urbana por ano**, comparada com um mapa de referência;
- **concordância pixel a pixel** com o mapa de referência.

Referência (`evaluation.reference`): por padrão, a classe de área urbanizada (24) da coleção pública do MapBiomas, acessível a qualquer pessoa. Nas tabelas, as colunas `col11` referem-se a esse mapa de referência.

As amostras de validação vêm de áreas estáveis, então a acurácia tende a ser otimista nas áreas de transição. Use-a para **comparar rodadas**, não como acurácia absoluta.

As cartas avaliadas seguem `scope` e `cartas_teste` em `config/col12.yaml`.

## 0 · Preparação

In [ ]:
import sys, os
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    !git clone -q -b col12 https://github.com/jpedrassoli/brazil-urban-area.git repo 2>/dev/null || (cd repo && git pull -q)
    %pip install -q -e "repo/lulc_30m_landsat/collection 12" geemap
    RAIZ = 'repo/lulc_30m_landsat/collection 12'
else:
    RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

import ee, pandas as pd
from mb_urban import auth, config, grids, evaluate, col11
cfg = config.load()

# --- Sua configuração -------------------------------------------------------------------------
# Usada quando não há um config/col12.yaml próprio: preencha com o SEU projeto do Cloud registrado no
# Earth Engine. A pasta de destino precisa existir; a tabela de amostras é o CSV de
# `collection 11/dados` subido no seu Earth Engine.
if config.config_path() == config.EXAMPLE_CONFIG:
    PROJETO = 'seu-projeto-do-cloud'
    cfg['cloud_project'] = PROJETO
    cfg['destinations']['teste'] = f'projects/{PROJETO}/assets/mb_urban_col12'
    cfg['samples']['table'] = f'projects/{PROJETO}/assets/amostras_col11'

## 1 · Conta do Earth Engine

**No Colab**, aparece um **link**: abra, entre com a sua conta do Earth Engine, autorize, copie o código que o Google mostra e cole no campo abaixo do bloco. O login usa o cliente do próprio Google Cloud (gcloud) e funciona também com contas institucionais. A credencial fica só na máquina temporária do Colab e some quando o ambiente é encerrado. **Não compartilhe o código de verificação.**

O aviso "Cannot find a quota project" pode ser ignorado: o projeto do Cloud é informado em seguida pelo notebook.

**No computador**, usa o login padrão do Earth Engine (só pede login se ainda não houver credencial).

In [ ]:
CONTA = 'default'
if CONTA == 'default':
    if EM_COLAB:
        # Login pelo gcloud (cliente de login do próprio Google). Aparece um link: abra, entre com a
        # sua conta, autorize, copie o código e cole no campo abaixo deste bloco. A credencial fica só
        # na máquina temporária do Colab. Não compartilhe o código com ninguém.
        adc = os.path.join(os.environ.get('CLOUDSDK_CONFIG', os.path.expanduser('~/.config/gcloud')),
                           'application_default_credentials.json')
        if not os.path.exists(adc):
            !gcloud auth application-default login --no-launch-browser --scopes=https://www.googleapis.com/auth/earthengine,https://www.googleapis.com/auth/cloud-platform
    else:
        ee.Authenticate()   # só pede login se ainda não houver credencial válida
auth.init(CONTA)

## 2 · Cartas da rodada
O mapa avaliado de cada carta é `<destino>/<postprocess.folder>/mapa_<carta>`.

In [ ]:
CARTAS = grids.cartas_in_scope(cfg)
SUFIXO = ''   # sufixo do nome dos mapas, se a rodada usou um

for c in CARTAS:
    print(c, '-', cfg['cartas_teste'].get(c, ''))

## 3 · Calcular as métricas
Pode levar alguns minutos por carta.

In [ ]:
resultados = []
for c in CARTAS:
    try:
        resultados.append(evaluate.evaluate_carta(c, cfg, suffix=SUFIXO))
        print('ok', c)
    except Exception as e:
        print('sem resultado para', c, '-', str(e)[:80])
resumo = evaluate.summary(resultados)
resumo.round(3)

## 4 · Área urbana ao longo do tempo (Col 12 × referência)

In [ ]:
import matplotlib.pyplot as plt
n = len(resultados)
fig, axs = plt.subplots((n + 2) // 3, 3, figsize=(14, 3.2 * ((n + 2) // 3)), squeeze=False)
for ax, r in zip(axs.flat, resultados):
    r['area'][['col12', 'col11']].plot(ax=ax, title=r['carta'])
    ax.set_ylabel('km²')
plt.tight_layout()

## 5 · Mapa de diferenças de uma carta
Amarelo: urbano na Col 12. Vermelho: urbano só na Col 12. Azul: urbano só na referência. Ligue e desligue as camadas no ícone de camadas do mapa. Se as duas versões forem iguais, vermelho e azul ficam vazios.

In [ ]:
import geemap
CARTA = CARTAS[0]; ANO = 2025
novo = evaluate.map_asset(CARTA, cfg, SUFIXO).select(f'classification_{ANO}')
velho = evaluate.reference_map(CARTA, cfg).select(f'classification_{ANO}')
m = geemap.Map(basemap='Esri.WorldImagery')  # fundo Esri: os mosaicos do OpenStreetMap bloqueiam o Colab; m.centerObject(grids.region(CARTA, cfg), 10)
m.addLayer(velho.selfMask(), {'palette': ['999999']}, 'referência', False)
m.addLayer(novo.selfMask(), {'palette': ['ffff00']}, 'Col 12 (urbano)')
m.addLayer(novo.eq(1).And(velho.eq(0)).selfMask(), {'palette': ['ff0000']}, 'só Col 12 (vermelho)')
m.addLayer(velho.eq(1).And(novo.eq(0)).selfMask(), {'palette': ['0000ff']}, 'só referência (azul)')
m.add_layer_control()
m

## 6 · Relatório HTML
Salva as tabelas num arquivo em `logs/relatorios/` (local, não vai para o GitHub).

In [ ]:
print(evaluate.html_report(resultados, cfg))